In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")
blindD = ak.from_parquet("Data_All_runs_BLINDED.parquet")

plt.close()


def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):

    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    #selector_proton = 'TightKMProtonSelection'
    #selector_pion = 'TightKMPionMicroSelection'
    #print(f"Now trying to create a mask with {selector_proton}")
    #print(f"Now trying to create a mask with {selector_pion}")
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2


pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")



for myPMask in ["SuperLooseKMProtonSelection"]:
    for myPiMask in ["SuperLooseKMPionMicroSelection"]: 

        indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                            lam1p_selector=myPMask, \
                            lam1pi_selector=myPiMask, \
                            lam2p_selector=myPMask, \
                            lam2pi_selector=myPiMask, \
                            verbosity=0)

        upperBound = 100
        
        tagFS = "Lambda0postFitFlightSignificance"
        tagFL = "Lambda0FlightLen"

        tag = tagFS

        if tag == tagFS:
            binCount = int(upperBound/10)
        elif tag == tagFL:
            binCount = int(upperBound/5)

        binCount = 60

        h = Hist.new.Reg(binCount, 0, upperBound, name=tag, label=r"Flight Sig") \
         .StrCat([], name="SP", label="SP modes", growth=True)\
         .Weight()

        for valu in ['998','1005']:
            spmask = (data['spmode']==valu)
            mes = data['BpostFitMes']
            de = data['BpostFitDeltaE']

            meslo = 5.27
            meshi = 5.3
            DeltaElo = -0.07
            DeltaEhi = 0.07

            mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 

            mask_mesDelE = ((mes<meslo) | (mes>meshi)) | ((de<DeltaElo) | (de>DeltaEhi))
            #mask_mesDelE = ((mes>meslo)&(mes<meshi)&(de>DeltaElo)&(de<DeltaEhi))
            mask_fit =  mask_pid & mask_mesDelE 
            mask_fit = mask_fit[spmask]


            lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask_fit]
            lamconflsig = lamconflsig[lamconflsig>0]
            lamfl_basic = data[spmask]['Lambda0FlightLen'][mask_fit]
            lamfl_basic = lamfl_basic[lamfl_basic>0]
            #numBary = data[spmask]['nB'][ak.all(mask_fit, axis=1)]
            nBCan = data[spmask]['nB']

            s = lamconflsig
            l = lamfl_basic
            eventnB = (nBCan==1)

            s = s[eventnB]
            l = l[eventnB]
            flightsig = []
            flightlen = []
            for event in s:
                for lam in event:
                    flightsig+=[lam]
            for event in l:
                for lam in event:
                    flightlen+=[lam]

            xS = flightsig
            xL = flightlen

            if tag == tagFS:
                x = xS
            elif tag == tagFL:
                x = xL


            scaled = False
            if scaled:
                if tag == tagFS:
                    stackMult = 3044/1710
                elif tag == tagFL:
                    stackMult = 3126/1774
            else:
                stackMult = 1

            #stackMult = 1939/1101

            if valu == '998':
                if tag == tagFL:
                    h.fill(Lambda0FlightLen=x, SP=valu,  weight=(424.3/1719)*stackMult)
                elif tag == tagFS:
                    h.fill(Lambda0postFitFlightSignificance=x, SP=valu,  weight=(424.3/1719)*stackMult)
                counts998,bins998 = np.histogram(x, bins=binCount, range=(0,upperBound))
            elif valu == '1005':
                if tag == tagFL:
                    h.fill(Lambda0FlightLen=x, SP=valu,  weight=(424.3/868.3)*stackMult)
                elif tag == tagFS:
                    h.fill(Lambda0postFitFlightSignificance=x, SP=valu,  weight=(424.3/868.3)*stackMult)
                counts1005,bins1005 = np.histogram(x, bins=binCount, range=(0,upperBound))

        

            
        print(sum(counts1005)*(424.3/868.3)+sum(counts998)*(424.3/1719))

        h.stack('SP')[:].project(tag).plot(stack=True,histtype="fill")
        
        indices_and_maps = get_info_for_PID_masks(blindD, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                            lam1p_selector=myPMask, \
                            lam1pi_selector=myPiMask, \
                            lam2p_selector=myPMask, \
                            lam2pi_selector=myPiMask, \
                            verbosity=0)
        
        #spmask = (blindD['spmode']=='-999')
        mes = blindD['BpostFitMes']
        de = blindD['BpostFitDeltaE']

        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 
        mask_mesDelE = ((mes<meslo) | (mes>meshi)) | ((de<DeltaElo) | (de>DeltaEhi))
        mask_fit = mask_pid & mask_mesDelE


        lamconflsig = blindD['Lambda0postFitFlightSignificance'][mask_fit]
        lamconflsig = lamconflsig[lamconflsig>0]
        lamfl_basic = blindD['Lambda0FlightLen'][mask_fit]
        lamfl_basic = lamfl_basic[lamfl_basic>0]
        nBCan = blindD['nB']
        eventnB = (nBCan==1)

        s = lamconflsig
        l = lamfl_basic

        s = s[eventnB]
        l = l[eventnB]
        flightsig = []
        flightlen = []
        for event in s:
            for lam in event:
                flightsig+=[lam]
        for event in l:
            for lam in event:
                flightlen+=[lam]

        xS = flightsig
        xL = flightlen

        if tag == tagFS:
            x = xS
        elif tag == tagFL:
            x = xL

        x = [val for val in x if val < upperBound]

        counts, bin_edges = np.histogram(x, bins=binCount, range = (0,upperBound))
        print(sum(counts))

        # 3. Find the center of each bin
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # 4. Calculate Poisson errors (standard statistical error for counts)
        errors = np.sqrt(counts)

        # 5. Overlay the points with error bars on top of the bars
        plt.errorbar(
            bin_centers, 
            counts, 
            yerr=errors, 
            fmt='o', 
            color='purple', 
            ecolor='purple', 
            capsize=3, 
            label='Data'
        )

        
        plt.xlabel(tag)
        plt.ylabel('Counts')
        plt.legend()
        plt.show()
